# Final exercise: Simple solution for the Ames housing dataset

In [1]:
import pandas as pd

house_prices = pd.read_csv("datasets/ames_housing_no_missing.csv")
house_prices.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,Grvl,Reg,Lvl,AllPub,Inside,...,0,Gd,MnPrv,Shed,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,Grvl,Reg,Lvl,AllPub,FR2,...,0,Gd,MnPrv,Shed,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,Grvl,IR1,Lvl,AllPub,Inside,...,0,Gd,MnPrv,Shed,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,Grvl,IR1,Lvl,AllPub,Corner,...,0,Gd,MnPrv,Shed,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,Grvl,IR1,Lvl,AllPub,FR2,...,0,Gd,MnPrv,Shed,0,12,2008,WD,Normal,250000


In [4]:
target_name = "SalePrice"
target = house_prices[target_name]
data = house_prices.drop(columns=[target_name])
data.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,RL,65.0,8450,Pave,Grvl,Reg,Lvl,AllPub,Inside,...,0,0,Gd,MnPrv,Shed,0,2,2008,WD,Normal
1,20,RL,80.0,9600,Pave,Grvl,Reg,Lvl,AllPub,FR2,...,0,0,Gd,MnPrv,Shed,0,5,2007,WD,Normal
2,60,RL,68.0,11250,Pave,Grvl,IR1,Lvl,AllPub,Inside,...,0,0,Gd,MnPrv,Shed,0,9,2008,WD,Normal
3,70,RL,60.0,9550,Pave,Grvl,IR1,Lvl,AllPub,Corner,...,0,0,Gd,MnPrv,Shed,0,2,2006,WD,Abnorml
4,60,RL,84.0,14260,Pave,Grvl,IR1,Lvl,AllPub,FR2,...,0,0,Gd,MnPrv,Shed,0,12,2008,WD,Normal


In [39]:
data.shape

(1460, 79)

In [6]:
from sklearn.compose import make_column_selector as selector

categorical_column_selector = selector(dtype_include=object)
numerical_columnn_selector = selector(dtype_exclude=object)

categorical_columns = categorical_column_selector(data)
numerical_columns = numerical_columnn_selector(data)

In [7]:
categorical_columns

['MSZoning',
 'Street',
 'Alley',
 'LotShape',
 'LandContour',
 'Utilities',
 'LotConfig',
 'LandSlope',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'BldgType',
 'HouseStyle',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Exterior2nd',
 'MasVnrType',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Heating',
 'HeatingQC',
 'CentralAir',
 'Electrical',
 'KitchenQual',
 'Functional',
 'FireplaceQu',
 'GarageType',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PavedDrive',
 'PoolQC',
 'Fence',
 'MiscFeature',
 'SaleType',
 'SaleCondition']

In [8]:
numerical_columns

['MSSubClass',
 'LotFrontage',
 'LotArea',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtFinSF1',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'MiscVal',
 'MoSold',
 'YrSold']

In [22]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

ordinal_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)
scaler = StandardScaler()

In [23]:
from sklearn.compose import make_column_transformer

preprocessor = make_column_transformer(
    (ordinal_encoder, categorical_columns),
    # (scaler, numerical_columns),
)

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", HistGradientBoostingRegressor(random_state=42)),
])

In [25]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "regressor__learning_rate": (0.001, 0.01, 0.1),
    "regressor__max_depth": (3, 5, 8),
    "regressor__max_features": (0.2, 0.4, 0.6, 0.8, 1.0),
}

model_grid_search = GridSearchCV(
    model,
    param_grid=param_grid,
)

In [40]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    model_grid_search,
    data,
    target,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=4,
    return_estimator=True,
    return_train_score=True,
)

In [43]:
cv_results["train_score"].mean()

np.float64(-19687.54675494327)

In [44]:
cv_results["test_score"].mean()

np.float64(-24839.244637331325)

In [45]:
for estimator in cv_results["estimator"]:
    print(estimator.best_params_)

{'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__max_features': 0.2}
{'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__max_features': 0.2}
{'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__max_features': 0.2}
{'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__max_features': 0.2}
{'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__max_features': 0.2}
